In [3]:
!pip install -q -U transformers bitsandbytes accelerate

In [4]:
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


#  we flush the gpu memory

In [5]:
gc.collect()
torch.cuda.empty_cache()

# we take the Qwen 2.5 model which is 1.5B parameters model

In [7]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

# we take the measure of vram before loading and load it as it is unquantized

In [8]:
print("1. Loading Unquantized Model (BF16)...")
vram_before_unquantized = torch.cuda.memory_allocated() / 1024**3

model_unquantized = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0} # Explicitly place model on GPU
)
vram_after_unquantized = torch.cuda.memory_allocated() / 1024**3
unquantized_vram_used = vram_after_unquantized - vram_before_unquantized
print(f"VRAM Used: {unquantized_vram_used:.2f} GB\n")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


1. Loading Unquantized Model (BF16)...


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

   ↳ Done. VRAM Used: 2.88 GB



# then we load the quantized version of the model and measure how much memory it takes

In [12]:
print("2. Loading Quantized Model (NF4 4-Bit)...")
vram_before_quantized = torch.cuda.memory_allocated() / 1024**3

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_quantized = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map={"": 0}
)

2. Loading Quantized Model (NF4 4-Bit)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [13]:
vram_after_quantized = torch.cuda.memory_allocated() / 1024**3
quantized_vram_used = vram_after_quantized - vram_after_unquantized
print(f"VRAM Used: {quantized_vram_used:.2f} GB\n")

   ↳ Done. VRAM Used: 1.07 GB



# we create a function to test out the inference on these both models unquantized and quantized

In [16]:
def run_inference(model, prompt_text):
    messages = [
        {"role": "system", "content": "You are a precise assistant. Extract: CATEGORY, PRIORITY (Low/High), and SUMMARY."},
        {"role": "user", "content": prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=False
        )

    # Strip prompt tokens from output
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()


# test the inferences on both the models

In [19]:
test_prompt = """
My tracking number is #992183.
The app says that the package is delivered but my front porch is empty.
There is nothing on it.
I need a replacement shipped out or a full refund immediately.
"""

print("=== RUNNING EXPERIMENT ===")
output_unquantized = run_inference(model_unquantized, test_prompt)
output_quantized = run_inference(model_quantized, test_prompt)

print(f"\n[UNQUANTIZED MODEL OUTPUT (BF16)]:\n{output_unquantized}")
print("-" * 50)
print(f"[QUANTIZED MODEL OUTPUT (NF4 4-BIT)]:\n{output_quantized}")
print("=" * 35)


=== RUNNING EXPERIMENT ===

[UNQUANTIZED MODEL OUTPUT (BF16)]:
- **CATEGORY:** Shipping Issue
- **PRIORITY:** High
- **SUMMARY:** The package was delivered according to the tracking number, but there is no item at the delivery address. The customer requests either a replacement shipment or a full refund.
--------------------------------------------------
[QUANTIZED MODEL OUTPUT (NF4 4-BIT)]:
**CATEGORY:** Shipping Issue  
**PRIORITY:** High  
**SUMMARY:** Package not delivered as expected; no item found at recipient's address. Request immediate action to replace shipment or issue a full refund.


# we can see here that the summaries are somewhat different but the meaning both hold is the same
# the category and the priority were identified correctly by both the models: unquantized and quantized as well

# next step we benchmark the memory footprints of these two models and calculate delta

In [23]:
vram_savings_percent = ((unquantized_vram_used - quantized_vram_used) / unquantized_vram_used) * 100

print("\n📈 QUANTIFIABLE LAB BENCHMARKS:")
print(f"• Unquantized Memory Footprint : {unquantized_vram_used:.2f} GB")
print(f"• Quantized Memory Footprint   : {quantized_vram_used:.2f} GB")
print(f"• Total Physical VRAM Savings  : {vram_savings_percent:.1f}% reduction")


📈 QUANTIFIABLE LAB BENCHMARKS:
• Unquantized Memory Footprint : 2.88 GB
• Quantized Memory Footprint   : 1.07 GB
• Total Physical VRAM Savings  : 62.6% reduction


# we can see we have achieved 60%+ reduction in our memory footprint. This contributes directly to the business value!